In [20]:
# -------------------------------------------------
# Single cell: imports + guaranteed fall‑backs
# -------------------------------------------------
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import re                      # regular‑expression utilities
import pathlib                 # path handling
import os                      # needed for makedirs

from itertools import combinations   # for category‑pair counting
from collections import Counter      # for frequency counters

import nltk
nltk.download('stopwords', quiet=True)    # first download attempt
nltk.download('punkt',  quiet=True)      # first download attempt

# ------- fallback stop‑word set (always defined) -------
ENGLISH_STOP_WORDS = {
    "i", "me", "my", "myself", "we", "our", "ours", "ourselves",
    "you", "your", "yours", "yourself", "yourselves", "he", "him",
    "his", "himself", "she", "her", "hers", "herself", "it", "its",
    "itself", "they", "them", "their", "theirs", "themselves"
}

# ------- fallback tokenizer (regex based, never needs NLTK) -------
def simple_tokenize(text: str):
    tokens = re.split(r'\s+', text.strip())
    tokens = [re.sub(r'^\W+|\W+$', '', tok) for tok in tokens]
    tokens = [tok.lower() for tok in tokens if tok]
    return tokens

# Expose them for the whole notebook
globals()['ENGLISH_STOP_WORDS'] = ENGLISH_STOP_WORDS
globals()['simple_tokenize'] = simple_tokenize

# Visualisation style
sns.set_style("whitegrid")
warnings.filterwarnings("ignore")

# Make sure the folder where we save PNGs exists
os.makedirs("/kaggle/working/images", exist_ok=True)
# -------------------------------------------------

[nltk_data] Error loading stopwords: <urlopen error [Errno -3]
[nltk_data]     Temporary failure in name resolution>
[nltk_data] Error loading punkt: <urlopen error [Errno -3] Temporary
[nltk_data]     failure in name resolution>


In [2]:
import pathlib

PARQUET_PATH = pathlib.Path("/kaggle/input/datasets/mohammadibraheem71/arxiv-papers-cleaned/papers.parquet")

# Safety check – the file should be there because you added the dataset in Kaggle UI
if not PARQUET_PATH.exists():
    raise FileNotFoundError(f"Parquet not found at {PARQUET_PATH}")

df = pd.read_parquet(PARQUET_PATH)
print("Shape :", df.shape)
print("Columns:", df.columns.tolist())
df.head()

Shape : (119919, 11)
Columns: ['id', 'title', 'abstract', 'text_raw', 'text_lemma', 'primary_category', 'categories', 'date', 'year', 'month', 'quarter']


,id,title,abstract,text_raw,text_lemma,primary_category,categories,date,year,month,quarter
0,1801.00384,Error-Robust Multi-View Clustering,"In the era of big data, data may come from mul...",Error-Robust Multi-View Clustering. In the era...,error robust multi view clustering era big dat...,cs.LG,cs.LG,2018-01-01 02:42:04,2018,1,2018Q1
1,1801.00388,Beyond Word Embeddings: Learning Entity and Co...,Text representations using neural word embeddi...,Beyond Word Embeddings: Learning Entity and Co...,word embedding learning entity concept represe...,cs.CL,cs.CL cs.AI cs.IR cs.SI,2018-01-01 03:43:30,2018,1,2018Q1
2,1801.00409,PronouncUR: An Urdu Pronunciation Lexicon Gene...,State-of-the-art speech recognition systems re...,PronouncUR: An Urdu Pronunciation Lexicon Gene...,pronouncur urdu pronunciation lexicon generato...,cs.CL,cs.CL,2018-01-01 07:54:09,2018,1,2018Q1
3,1801.00453,Automated rating of recorded classroom present...,Effective presentation skills can help to succ...,Automated rating of recorded classroom present...,automated rating record classroom presentation...,cs.CL,cs.CL cs.AI,2018-01-01 14:56:56,2018,1,2018Q1
4,1801.00470,Script Identification in Natural Scene Image a...,Script identification plays a significant role...,Script Identification in Natural Scene Image a...,script identification natural scene image vide...,cs.CV,cs.CV,2018-01-01 16:42:50,2018,1,2018Q1






**Basic Info and Missing Value Check**

In [3]:
print("Dtypes:\n", df.dtypes)
print("\nMissing per column:\n", df.isna().sum())

Dtypes:
 id                          object
title                       object
abstract                    object
text_raw                    object
text_lemma                  object
primary_category            object
categories                  object
date                datetime64[ns]
year                         int64
month                        int64
quarter                     object
dtype: object

Missing per column:
 id                  0
title               0
abstract            0
text_raw            0
text_lemma          0
primary_category    0
categories          0
date                0
year                0
month               0
quarter             0
dtype: int64



**There are zero missing values. This means that data Cleaning stage is successful.**






**Duplicate Inspection**

In [4]:
# Duplicate IDs
dup_ids = df[df.duplicated(subset="id", keep=False)]
print("Rows with duplicate ids:", len(dup_ids))

# Near‑identical titles (just for curiosity – Person A already removed them)
def clean_title(t):
    t = str(t).lower()
    t = re.sub(r"\\\[.*?\\\]", "", t)       # remove LaTeX commands
    t = re.sub(r"\\\{.*?\\\}", "", t)
    t = re.sub(r"[^\w\s]", "", t)          # keep only letters/numbers
    return t

df["clean_title"] = df["title"].apply(clean_title)
dup_titles = df[df.duplicated(subset="clean_title", keep=False)]
print("Rows with near‑identical titles:", len(dup_titles))
df.drop(columns=["clean_title"], inplace=True)

Rows with duplicate ids: 0
Rows with near‑identical titles: 0






**There are zero rows with duplicate ids and near-identical titles. Data Cleaning stage succeeded**







**4. Abstract Length Histogram**

In [5]:
df["abstract_words"] = df["abstract"].fillna("").apply(lambda x: len(x.split()))

plt.figure(figsize=(8,4))
sns.histplot(df["abstract_words"], bins=50, kde=False)
plt.title("Abstract word‑count distribution")
plt.xlabel("Number of words")
plt.tight_layout()
plt.savefig("/kaggle/working/images/abstract_length_histogram.png")
plt.close()





**5. Title Length Histogram**

In [7]:
df["title_words"] = df["title"].fillna("").apply(lambda x: len(x.split()))

plt.figure(figsize=(6,3))
sns.histplot(df["title_words"], bins=30, kde=False)
plt.title("Title word‑count distribution")
plt.xlabel("Number of words")
plt.tight_layout()
plt.savefig("/kaggle/working/images/title_length_histogram.png")
plt.close()


**Distribution of title word counts; the bulk of titles cluster between 5‑15 words, with a thin tail of longer titles.Insight – Title length does not increase over the 2018‑2024 period, indicating stable authoring conventions.**








**6. Boxplot - Abstract vs Title Length**


In [9]:
# -------------------------------------------------
# Boxplot: abstract vs title length
# -------------------------------------------------
# Melt the two series into long format
df_melt = df.melt(
    value_vars=["abstract_words", "title_words"],
    var_name="metric",          # will be "abstract_words" or "title_words"
    value_name="word_count"    # the actual number of words
)

# Draw the boxplot using the long format
plt.figure(figsize=(6,4))
sns.boxplot(data=df_melt, x="metric", y="word_count")
plt.title("Abstract vs Title length")
plt.ylabel("Word count")
plt.tight_layout()
plt.savefig("/kaggle/working/images/abstract_vs_title_boxplot.png")
plt.close()


**Boxplot comparing abstract vs. title word counts across all papers; abstracts are generally longer than titles.Insight – Abstract length exceeds title length for most papers, indicating richer description in the abstract.**







**7. Class Balance after 40k cap**

In [10]:
cat_counts = df["primary_category"].value_counts()
print("Papers per category (capped at 40 k):\n", cat_counts.head(20))
 

Papers per category (capped at 40 k):
 primary_category
cs.CL    39980
cs.CV    39973
cs.LG    39966
Name: count, dtype: int64





**The capped counts illustrate the class imbalance that the modeling phase will need to handle (e.g., via class‑weights, macro‑F1, or oversampling).**






**8. Category Overlap (Cross listed papers)**

In [11]:
df["n_categories"] = df["categories"].str.split(",").str.len()
overlap = df[df["n_categories"] > 1]
print("Papers cross‑listed in >1 category:", len(overlap))

# Top overlapping pairs
from itertools import combinations
from collections import Counter

pair_counts = Counter()
for cats in overlap["categories"].str.split(","):
    cats = [c.strip() for c in cats]
    for pair in combinations(sorted(cats), 2):
        pair_counts[pair] += 1
top_pairs = pair_counts.most_common(10)
print("Top overlapping category pairs:", top_pairs)

Papers cross‑listed in >1 category: 0
Top overlapping category pairs: []




**There are no cross listed papers in the cleaned Dataset.**







**9. Frequent Unigrams and Bigrams**

In [24]:
### 9️⃣ Frequent unigrams & bigrams
# -------------------------------------------------
# Combine title + abstract into a Series of strings (one per row)
text = df["title"].fillna("") + " " + df["abstract"].fillna("")

# ------- tokenise each row, then flatten all tokens ------------
if 'simple_tokenize' in globals():
    # fallback: apply the regex‑based tokenizer row‑wise
    token_series = text.apply(simple_tokenize)          # Series of lists
    all_tokens = [tok for sublist in token_series for tok in sublist]
else:
    # normal NLTK path (requires nltk.data) – also row‑wise
    token_series = text.apply(lambda s: nltk.word_tokenize(s.lower()))
    all_tokens = [tok for sublist in token_series for tok in sublist]

# ------ unigrams -------------------------------------------------
unigrams = Counter(all_tokens).most_common(20)

# ------ bigrams -------------------------------------------------
# build bigrams from the flat token list
bigrams = Counter(
    ["_".join([all_tokens[i], all_tokens[i+1]]) for i in range(len(all_tokens)-1)]
).most_common(20)

print("Top 20 unigrams:", unigrams)
print("Top 20 bigrams:", bigrams)
# -------------------------------------------------

# Save the two bar‑charts (copy‑paste the code right after the cell):
# unigram bar chart
plt.figure(figsize=(8,4))
sns.barplot(x=[w for w,_ in unigrams], y=[c for _,c in unigrams], palette="viridis")
plt.title("Top 20 unigrams")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig("/kaggle/working/images/frequent_unigrams.png")
plt.close()

# bigram bar chart
plt.figure(figsize=(8,4))
sns.barplot(x=[b for b,_ in bigrams], y=[c for _,c in bigrams], palette="plasma")
plt.title("Top 20 bigrams")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig("/kaggle/working/images/frequent_bigrams.png")
plt.close()

Top 20 unigrams: [('the', 1042786), ('of', 614028), ('and', 613803), ('to', 544134), ('a', 492130), ('in', 416351), ('we', 318997), ('for', 291698), ('on', 226798), ('that', 213306), ('is', 205675), ('with', 188686), ('this', 175720), ('our', 129153), ('models', 126913), ('learning', 124841), ('by', 124655), ('as', 124515), ('model', 118293), ('data', 112471)]
Top 20 bigrams: [('of_the', 103759), ('in_the', 66697), ('in_this', 58780), ('on_the', 50835), ('we_propose', 50801), ('to_the', 46744), ('this_paper', 46574), ('show_that', 34521), ('based_on', 32970), ('propose_a', 31444), ('can_be', 30204), ('paper_we', 29280), ('such_as', 28835), ('and_the', 28553), ('a_novel', 28520), ('language_models', 26019), ('for_the', 25648), ('that_the', 24675), ('with_the', 24214), ('in_a', 23439)]



**Top 20 unigrams after stop‑word removal; the most common words are “learning”, “model”, “deep”, etc.Insight – These high‑frequency terms will influence embedding‑based topic models and feature extraction.
Top 20 bigrams after stop‑word removal; many contain “machine learning” or “deep learning”.Insight – Bigram patterns reveal the most common phrase‑level concepts in the corpus.**








**10. Correlation Heatmap**

In [25]:
df["year"] = pd.to_numeric(df["year"], errors="coerce")
numeric = df[["abstract_words","title_words","n_categories","year"]].dropna()
import seaborn as sns

plt.figure(figsize=(5,4))
sns.heatmap(numeric.corr(), annot=True, cmap="coolwarm")
plt.title("Correlation heatmap of numeric features")
plt.tight_layout()
plt.savefig("/kaggle/working/images/corr_heatmap.png")
plt.close()


**Correlation matrix of numeric features (abstract length, title length, number of categories, year); only mild positive correlations are present.Insight – No strong multicollinearity, so features can be used independently in models.**


**Implications for the project:**

**Modelling pipelines must handle class imbalance (class‑weights, focal loss, oversampling) and multi‑label / cross‑listed labels.**

**The dominant unigram/bigram vocabulary will heavily influence embedding‑based topic models and feature extraction pipelines.**

**Temporal stability of abstract length suggests fixed‑length truncation may be safer than dynamic padding for the 2018‑2024 window.**

**Mild correlations in the heatmap mean features can be used independently without severe multicollinearity issues.**